# 50 구조화 조회 계약과 재정렬 입력 경계 검사

목적: 상세 설명 요청 때문에 구조화 조회를 건너뛰는 문제, 잘못된 대상 일치와 카드 범위 문제를 실험 경로에서 고친다. 운영 DB·코드·OCR은 수정하지 않는다. API 호출은 없다.

공유 SQL 계약과 회귀 검사는 51–54에서 동일하게 재사용해야 하므로 `scripts/50_query_contract.py`와 `scripts/50_test_query_contract.py`로 분리했다. 실행·원본 확인·판정은 이 노트북에 남긴다. HTML은 결과 확인 후 직접 작성한다.

In [1]:
from pathlib import Path
import importlib.util, sys, json, hashlib, time, resource, subprocess
ROOT=Path.cwd()
if not (ROOT/'notebooks').exists(): ROOT=ROOT.parent
sys.dont_write_bytecode=True
spec=importlib.util.spec_from_file_location('query50',ROOT/'notebooks/scripts/50_query_contract.py')
Q=importlib.util.module_from_spec(spec);sys.modules[spec.name]=Q;spec.loader.exec_module(Q)
PATH=ROOT/'notebooks/data/50_query_contract_audit'
Q.P.F.STAGES[50]=PATH;Q.P.F.initialize(50)
def write(name,value): Q.P.F.write_json(PATH/name,value)
def frozen(name,value):
 p=PATH/name
 if p.exists(): assert json.loads(p.read_text())==value, '동결한 입력 변경: '+name
 else: write(name,value)
started=time.perf_counter();cpu_started=time.process_time()
prior=json.loads((ROOT/'notebooks/data/49_selected_pipeline_confirmation/questions.json').read_text())['queries']
extra=[
('D33','BC 카드 중에서 커피 할인 혜택 있는 카드 찾아줘.','scope'),
('D34','BC 카드 중에서 전월실적 30만원 이하이고 넷플릭스 혜택 있는 카드 찾아줘.','scope'),
('D35','BC 카드 중에서 커피와 넷플릭스 혜택 있는 카드 찾아줘.','scope'),
('D36','커피 할인 카드의 월 한도와 횟수, 전월실적을 알려줘.','detail'),
('D37','커피 할인 월 한도가 1만원 이상인 카드 찾아줘.','numeric'),
('D38','커피 할인을 월 3회 이상 받을 수 있는 카드 찾아줘.','numeric'),
('D39','철도, 고속버스, 택시를 함께 할인하는 카드 찾아줘.','semantic'),
('D40','기차, 고속버스, 택시를 함께 할인하는 카드 찾아줘.','semantic')]
queries=prior+[dict(query_id=i,question=q,subgroup=g) for i,q,g in extra]
frozen('questions.json',dict(queries=queries,split='development',source49='49_selected_pipeline_confirmation/questions.json'))
# 52번 결과를 보기 전에 53번 질문을 고정한다.
frozen('53_preselected_questions.json',dict(query_ids=['H08','H11','H18','H21','H01','H02','H03','H13','H15','H22','H23','D36'],reason='혜택·상세조건·같은 카드 복합·발급사 범위를 포함한 원문 확인 대상'))
result=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'notebooks/scripts/50_test_query_contract.py'),'--tb=short'],cwd=ROOT,capture_output=True,text=True)
write('green_regressions.json',dict(command=result.args,exit_code=result.returncode,stdout=result.stdout,stderr=result.stderr))
print(result.stdout)
assert result.returncode==0
print('개발 질문',len(queries),'API 호출 0')

............                                                             [100%]
12 passed in 0.59s

개발 질문 40 API 호출 0


In [2]:
import fitz
facts=Q.facts_by_id()
cases=[('H08','fact_e1da0e4696793ed908db92b5',4,['20','3,000','2회']),
       ('H11','fact_e43bb1b1445a02ca3ec50122',2,['7천원','5천원','1회']),
       ('H21','fact_b1dcc4598090e0ee2af4f17d',2,['철도','고속버스','택시'])]
audits=[]
for qid,fid,page,terms in cases:
 f=facts[fid]; pdf=ROOT/'data/raw'/(f['document_id']+'.pdf')
 with fitz.open(pdf) as doc:
  texts=[p.get_text() for p in doc]
  positions=[i+1 for i,t in enumerate(texts) if all(term.replace(' ','') in ''.join(t.split()) for term in terms)]
  selected=texts[page-1] if page else '\n'.join(texts)
  found=all(term.replace(' ','') in ''.join(selected.split()) for term in terms)
  if qid=='H21':
   visual=json.loads((PATH/'H21_visual_source_review.json').read_text())
   assert visual['passed'] and visual['page']==page and hashlib.sha256((PATH/'H21_pdf_page2.png').read_bytes()).hexdigest()==visual['image_sha256']
   found=True; selected='PDF 텍스트층 없음. H21_visual_source_review.json의 직접 이미지 확인 기록과 H21_pdf_page2.png 참조.'; positions=[2]
 audits.append(dict(query_id=qid,fact_id=fid,source=str(pdf.relative_to(ROOT)),source_sha256=hashlib.sha256(pdf.read_bytes()).hexdigest(),checked_page=page,matching_pages=positions,terms=terms,source_terms_present=found,fields=f['fields'],source_text=selected,classification='source_and_DB_present' if found else 'needs_source_review'))
write('source_audit.json',dict(records=audits,notes=['H08 저장 cap_period는 unknown이지만 cap 원문은 통합 월 3,000원. 읽기 시 명시 기간만 해석한다.','H11 상세 조건 요청은 unsupported_filter가 아니다.','H21 철도는 저장됨. 기차→철도 별칭은 추가하지 않음.']))
print([(a['query_id'],a['source_terms_present'],a['matching_pages']) for a in audits])
assert all(a['source_terms_present'] for a in audits)
# 요청 역할에 따른 실제 SQL과 상세/요약 일치 행 모두 보존
probes=[]
for qid,scope,requirements,fields in [
 ('H08',['BC/BC_Green_v2'],[Q.requirement(['커피'],action='discount')],['frequency','condition','cap']),
 ('H11',None,[Q.requirement(['넷플릭스'],action='discount',min_percent=20),Q.requirement(['유튜브'],action='discount',min_percent=20)],['condition','frequency']),
 ('H21',None,[Q.requirement(['기차'],action='discount'),Q.requirement(['고속버스'],action='discount'),Q.requirement(['택시'],action='discount')],['exceptions']),
 ('D39',None,[Q.requirement(['철도'],action='discount'),Q.requirement(['고속버스'],action='discount'),Q.requirement(['택시'],action='discount')],[])]:
 probes.append(dict(query_id=qid,result=Q.query(Q.plan(requirements,fields),scope)))
write('sql_probes.json',dict(records=probes))
print([(p['query_id'],len(p['result']['eligible_cards'])) for p in probes])

[('H08', True, [3, 4]), ('H11', True, [2]), ('H21', True, [2])]
[('H08', 1), ('H11', 6), ('H21', 0), ('D39', 1)]


In [3]:
# 원문 필드/예외/출처를 자르지 않은 모든 사실의 BGE 입력 길이 검사
B=Q.P.load('stage50_token_guard','34_bundle_experiment.py')
B.F.STAGES[50]=PATH;B.F.initialize(50)
guard=B.TokenGuard()
# 실제 개발 질문 중 토큰 수가 가장 긴 질문으로 모든 문서/쌍 길이를 검사
question=max((q['question'] for q in queries),key=lambda q:len(guard.tokenizer(q,add_special_tokens=False)['input_ids']))
lengths=[]
for fact in facts.values():
 text=Q.serialize_fact(fact)
 lengths.append(dict(fact_id=fact['fact_id'],card_key=fact['document_id'],bytes=len(text.encode()),**guard.lengths(question,text)))
violations=[r for r in lengths if r['document_tokens']>guard.document_limit or r['pair_tokens']>guard.max_length]
write('whole_fact_token_audit.json',dict(records=lengths,count=len(lengths),document_limit=guard.document_limit,pair_limit=guard.max_length,longest_question=question,violations=violations,truncation=False))
print('전체 사실',len(lengths),'문서 최대',max(r['document_tokens'] for r in lengths),'쌍 최대',max(r['pair_tokens'] for r in lengths),'경계 초과',len(violations))
before=json.loads((PATH/'preservation_before.json').read_text());changes=[]
for name,old in before.items():
 p=ROOT/name
 if not p.exists() or p.stat().st_size!=old['size'] or p.stat().st_mtime_ns!=old['mtime_ns'] or hashlib.sha256(p.read_bytes()).hexdigest()!=old['sha256']: changes.append(name)
write('preservation_after.json',dict(checked_files=len(before),changes=changes))
status=dict(stage=50,status='complete' if not violations and not changes else 'blocked',paid_stage_allowed=not violations and not changes,
 regressions_passed=result.returncode==0,source_audit_passed=all(a['source_terms_present'] for a in audits),whole_fact_count=len(lengths),token_violations=len(violations),preservation_changes=changes,
 wall_seconds=time.perf_counter()-started,cpu_seconds=time.process_time()-cpu_started,peak_rss_kib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
 API_requests=0,cost_usd='0',first_token_seconds=None,answer_output_seconds=None,time_note='오프라인 실행 시간. LLM 답변 출력 시간은 다음 유료 단계에서 별도 계측.')
write('completion_status.json',status)
print(json.dumps(status,ensure_ascii=False,indent=2))

전체 사실 3403 문서 최대 1601 쌍 최대 1649 경계 초과 0
{
  "stage": 50,
  "status": "complete",
  "paid_stage_allowed": true,
  "regressions_passed": true,
  "source_audit_passed": true,
  "whole_fact_count": 3403,
  "token_violations": 0,
  "preservation_changes": [],
  "wall_seconds": 11.2301255689963,
  "cpu_seconds": 8.847571069999999,
  "peak_rss_kib": 949956,
  "API_requests": 0,
  "cost_usd": "0",
  "first_token_seconds": null,
  "answer_output_seconds": null,
  "time_note": "오프라인 실행 시간. LLM 답변 출력 시간은 다음 유료 단계에서 별도 계측."
}
